# Notebook 07 — Construcción de `silver.mosaic`

## Objetivo

Construir la tabla `silver.mosaic` a partir de `bronze.mosaic`, que contiene información geodemográfica (Experian Mosaic) para 6.457 códigos postales españoles. La tabla bronze original tiene 76 columnas, mayoritariamente proporciones por segmento fino que no se utilizarán en el análisis principal del TFG. Por ello, en Silver se conservan únicamente las **siete variables clave** y se añaden **cinco flags binarios** que codifican lecturas comerciales de los grupos MOSAIC para Selmark.

## Transformaciones aplicadas

1. **Selección de columnas**: se reduce de 76 a 12 columnas finales (7 informativas + 5 flags derivados).
2. **Normalización del código postal**: aplicación de `LPAD(CP, 5, '0')` para corregir los 1.243 registros que perdieron el cero inicial durante la carga.
3. **Casteo numérico**: las columnas `Renta_Media`, `Max_Mosaic1` y `Max_Mosaic2` se cargaron como VARCHAR debido a la presencia de cadenas 'NaN'. Se aplica `TRY_CAST` a DOUBLE, convirtiendo los valores no numéricos en NULL.
4. **Flags comerciales**: se generan cinco variables binarias (`perfil_premium`, `perfil_familiar_joven`, `perfil_turistico`, `perfil_rural`, `perfil_precio_sensible`) basadas en la lectura comercial de los grupos MOSAIC documentada en el notebook 02.
5. **Validación de cobertura**: se simula el cruce con `silver.dim_cliente` para confirmar el porcentaje de clientes nacionales con enriquecimiento MOSAIC.

## 1. Configuración y conexión a DuckDB

In [2]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb


## 2. Verificación del esquema de `bronze.mosaic`

Antes de construir Silver, verificamos que los nombres de las columnas clave coinciden con los esperados (`CP`, `PROV_INE`, `Max_Mosaic_G`, `Max_Mosaic`, `Max_Mosaic1`, `Max_Mosaic2`, `Renta_Media`).

In [3]:
print("=" * 60)
print("COLUMNAS CLAVE EN bronze.mosaic")
print("=" * 60)

# Total de columnas
total_cols = con.execute("""
    SELECT COUNT(*) FROM information_schema.columns
    WHERE table_schema = 'bronze' AND table_name = 'mosaic'
""").fetchone()[0]
print(f"Total de columnas: {total_cols}\n")

# Buscar las columnas que necesitamos
print("Columnas relevantes detectadas:")
relevantes = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze' AND table_name = 'mosaic'
      AND column_name IN ('CP', 'PROV_INE', 'Max_Mosaic_G', 'Max_Mosaic', 
                          'Max_Mosaic1', 'Max_Mosaic2', 'Renta_Media')
    ORDER BY column_name
""").fetchdf()
print(relevantes.to_string(index=False))

print(f"\nColumnas relevantes encontradas: {len(relevantes)}/7")

COLUMNAS CLAVE EN bronze.mosaic
Total de columnas: 76

Columnas relevantes detectadas:
 column_name data_type
          CP   VARCHAR
  Max_Mosaic   VARCHAR
 Max_Mosaic1   VARCHAR
 Max_Mosaic2   VARCHAR
Max_Mosaic_G   VARCHAR
    PROV_INE   VARCHAR
 Renta_Media   VARCHAR

Columnas relevantes encontradas: 7/7


## 3. Construcción de `silver.mosaic`

Se crea la tabla aplicando todas las transformaciones de una sola sentencia `CREATE OR REPLACE TABLE`: selección de columnas, normalización del código postal, casteo numérico de las variables `Renta_Media`, `Max_Mosaic1` y `Max_Mosaic2`, y generación de los cinco flags comerciales derivados del grupo MOSAIC dominante.

In [4]:
print("Construyendo silver.mosaic...\n")

con.execute("""
    CREATE OR REPLACE TABLE silver.mosaic AS
    SELECT
        -- Código postal normalizado a 5 dígitos
        LPAD(TRIM(CP), 5, '0') AS codigo_postal_norm,
        
        -- Provincia
        TRIM(PROV_INE) AS provincia_ine,
        
        -- Grupo y peso del grupo dominante
        TRIM(Max_Mosaic_G) AS mosaic_grupo,
        TRY_CAST(Max_Mosaic2 AS DOUBLE) AS mosaic_grupo_peso,
        
        -- Segmento fino y peso del segmento dominante
        TRIM(Max_Mosaic) AS mosaic_segmento,
        TRY_CAST(Max_Mosaic1 AS DOUBLE) AS mosaic_segmento_peso,
        
        -- Renta media (TRY_CAST resuelve los 'NaN' devolviendo NULL)
        TRY_CAST(Renta_Media AS DOUBLE) AS renta_media,
        
        -- ---- FLAGS COMERCIALES DERIVADOS (basados en notebook 02) ----
        
        -- Perfil premium: A (Élites) + C (Éxito Provincial)
        CASE WHEN TRIM(Max_Mosaic_G) IN ('A', 'C') THEN TRUE ELSE FALSE END AS perfil_premium,
        
        -- Perfil familiar joven: D (Juventud en Expansión) + E (Profesionales Maduros)
        CASE WHEN TRIM(Max_Mosaic_G) IN ('D', 'E') THEN TRUE ELSE FALSE END AS perfil_familiar_joven,
        
        -- Perfil turístico: F (Turismo)
        CASE WHEN TRIM(Max_Mosaic_G) = 'F' THEN TRUE ELSE FALSE END AS perfil_turistico,
        
        -- Perfil rural: J (Agricultura) + K (Áreas Pasivas)
        CASE WHEN TRIM(Max_Mosaic_G) IN ('J', 'K') THEN TRUE ELSE FALSE END AS perfil_rural,
        
        -- Perfil precio-sensible: H (Áreas Mixtas) + I (No Cualificados) + K (Áreas Pasivas)
        CASE WHEN TRIM(Max_Mosaic_G) IN ('H', 'I', 'K') THEN TRUE ELSE FALSE END AS perfil_precio_sensible
        
    FROM bronze.mosaic
""")

print("✅ silver.mosaic creada correctamente")

Construyendo silver.mosaic...

✅ silver.mosaic creada correctamente


## 4. Validación de la tabla resultante

In [5]:
print("=" * 60)
print("AUDITORÍA DE VOLÚMENES Y ESQUEMA")
print("=" * 60)

n_bronze = con.execute("SELECT COUNT(*) FROM bronze.mosaic").fetchone()[0]
n_silver = con.execute("SELECT COUNT(*) FROM silver.mosaic").fetchone()[0]
print(f"Filas en bronze : {n_bronze:>8,}")
print(f"Filas en silver : {n_silver:>8,}")
print(f"% conservado    : {n_silver/n_bronze*100:>8.2f} %")

print("\n" + "=" * 60)
print("ESQUEMA DE silver.mosaic")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'mosaic'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))

AUDITORÍA DE VOLÚMENES Y ESQUEMA
Filas en bronze :    6,457
Filas en silver :    6,457
% conservado    :   100.00 %

ESQUEMA DE silver.mosaic
           column_name data_type
    codigo_postal_norm   VARCHAR
         provincia_ine   VARCHAR
          mosaic_grupo   VARCHAR
     mosaic_grupo_peso    DOUBLE
       mosaic_segmento   VARCHAR
  mosaic_segmento_peso    DOUBLE
           renta_media    DOUBLE
        perfil_premium   BOOLEAN
 perfil_familiar_joven   BOOLEAN
      perfil_turistico   BOOLEAN
          perfil_rural   BOOLEAN
perfil_precio_sensible   BOOLEAN


In [6]:
print("=" * 60)
print("VALIDACIÓN DEL CÓDIGO POSTAL NORMALIZADO")
print("=" * 60)
calidad_cp = con.execute("""
    SELECT 
        SUM(CASE WHEN LENGTH(codigo_postal_norm) = 5 THEN 1 ELSE 0 END) AS cps_5_digitos,
        SUM(CASE WHEN LENGTH(codigo_postal_norm) <> 5 THEN 1 ELSE 0 END) AS cps_anomalos,
        SUM(CASE WHEN codigo_postal_norm IS NULL THEN 1 ELSE 0 END) AS cps_nulos,
        COUNT(DISTINCT codigo_postal_norm) AS cps_distintos
    FROM silver.mosaic
""").fetchdf()
print(calidad_cp.to_string(index=False))

print("\n" + "=" * 60)
print("CALIDAD DE renta_media (tras TRY_CAST)")
print("=" * 60)
calidad_renta = con.execute("""
    SELECT 
        COUNT(*) AS total,
        SUM(CASE WHEN renta_media IS NULL THEN 1 ELSE 0 END) AS nulos,
        ROUND(MIN(renta_media), 2) AS minimo,
        ROUND(MAX(renta_media), 2) AS maximo,
        ROUND(AVG(renta_media), 2) AS media,
        ROUND(MEDIAN(renta_media), 2) AS mediana
    FROM silver.mosaic
""").fetchdf()
print(calidad_renta.to_string(index=False))

VALIDACIÓN DEL CÓDIGO POSTAL NORMALIZADO
 cps_5_digitos  cps_anomalos  cps_nulos  cps_distintos
        6457.0           0.0        0.0           6457

CALIDAD DE renta_media (tras TRY_CAST)
 total  nulos  minimo  maximo    media  mediana
  6457  626.0  2370.0 37743.0 23594.02  24111.0


In [7]:
print("=" * 60)
print("DISTRIBUCIÓN POR GRUPO MOSAIC")
print("=" * 60)
por_grupo = con.execute("""
    SELECT 
        mosaic_grupo,
        COUNT(*) AS num_cps,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje,
        ROUND(AVG(renta_media), 0) AS renta_media_grupo
    FROM silver.mosaic
    GROUP BY mosaic_grupo
    ORDER BY num_cps DESC
""").fetchdf()
print(por_grupo.to_string(index=False))

DISTRIBUCIÓN POR GRUPO MOSAIC
mosaic_grupo  num_cps  porcentaje  renta_media_grupo
           K     1971       30.53            21852.0
           H     1452       22.49            23151.0
           C      694       10.75            26771.0
           J      507        7.85            19122.0
           I      500        7.74            20701.0
           A      295        4.57            31908.0
           D      236        3.65            26517.0
           B      225        3.48            27031.0
           G      224        3.47            26057.0
           E      182        2.82            28441.0
           F      128        1.98            24845.0
           U       43        0.67            22266.0


In [8]:
print("=" * 60)
print("VALIDACIÓN DE FLAGS COMERCIALES DERIVADOS")
print("=" * 60)
flags = con.execute("""
    SELECT 
        SUM(CASE WHEN perfil_premium THEN 1 ELSE 0 END) AS perfil_premium,
        SUM(CASE WHEN perfil_familiar_joven THEN 1 ELSE 0 END) AS perfil_familiar_joven,
        SUM(CASE WHEN perfil_turistico THEN 1 ELSE 0 END) AS perfil_turistico,
        SUM(CASE WHEN perfil_rural THEN 1 ELSE 0 END) AS perfil_rural,
        SUM(CASE WHEN perfil_precio_sensible THEN 1 ELSE 0 END) AS perfil_precio_sensible,
        COUNT(*) AS total_cps
    FROM silver.mosaic
""").fetchdf()
print(flags.T.to_string(header=False))

VALIDACIÓN DE FLAGS COMERCIALES DERIVADOS
perfil_premium           989.0
perfil_familiar_joven    418.0
perfil_turistico         128.0
perfil_rural            2478.0
perfil_precio_sensible  3923.0
total_cps               6457.0


In [9]:
print("=" * 60)
print("COBERTURA AL CRUZAR CON silver.dim_cliente")
print("=" * 60)

cobertura = con.execute("""
    SELECT 
        COUNT(*) AS clientes_total,
        SUM(CASE WHEN d.tipo_mercado = 'NACIONAL' THEN 1 ELSE 0 END) AS clientes_nacionales,
        SUM(CASE WHEN d.tipo_mercado = 'NACIONAL' AND m.codigo_postal_norm IS NOT NULL THEN 1 ELSE 0 END) AS nacionales_con_mosaic,
        ROUND(
            100.0 * SUM(CASE WHEN d.tipo_mercado = 'NACIONAL' AND m.codigo_postal_norm IS NOT NULL THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN d.tipo_mercado = 'NACIONAL' THEN 1 ELSE 0 END), 0),
            2
        ) AS cobertura_pct
    FROM silver.dim_cliente d
    LEFT JOIN silver.mosaic m ON d.codigo_postal_norm = m.codigo_postal_norm
""").fetchdf()
print(cobertura.T.to_string(header=False))

COBERTURA AL CRUZAR CON silver.dim_cliente
clientes_total         3469.00
clientes_nacionales    1869.00
nacionales_con_mosaic  1820.00
cobertura_pct            97.38


## 5. Conclusiones del notebook 07

### Resumen del proceso

La tabla `silver.mosaic` se ha construido a partir de `bronze.mosaic` reduciendo la estructura original de 76 columnas a 12 (7 informativas + 5 flags comerciales derivados). Se ha conservado la totalidad del volumen (6.457 códigos postales, 100%) y se han aplicado tres transformaciones principales: normalización del código postal a 5 dígitos, casteo numérico de las variables socioeconómicas, y generación de flags comerciales derivados de la lectura de los grupos MOSAIC.

### Validación de las transformaciones aplicadas

**Normalización del código postal**: la totalidad de los 6.457 registros presenta ahora `codigo_postal_norm` con exactamente 5 dígitos, sin nulos, y todos los valores son únicos. La aplicación de `LPAD(CP, 5, '0')` ha resuelto íntegramente la incidencia detectada en el notebook 02 (1.243 CPs sin cero inicial).

**Casteo numérico de `renta_media`**: el `TRY_CAST` ha convertido correctamente las cadenas numéricas en valores DOUBLE y ha tratado los 626 valores 'NaN' como NULL (9,7% del total). Los valores resultantes oscilan entre 2.370 € y 37.743 €, con una media de 23.594 € y mediana de 24.111 €, coherente con la realidad socioeconómica española.

**Distribución por grupo MOSAIC**: la distribución se mantiene exacta respecto al análisis exploratorio previo: el grupo K (Áreas Pasivas) concentra el 30,5% de los códigos postales y el grupo H (Áreas Mixtas) el 22,5%, sumando entre ambos más de la mitad del territorio cubierto. Los grupos premium A y C, con renta media superior, representan conjuntamente el 15,3% del territorio. La renta media por grupo refleja la jerarquía esperada: A (31.908 €) > E (28.441 €) > B (27.031 €) > D (26.517 €) > C (26.771 €) en la franja alta; J (19.122 €) > I (20.701 €) > K (21.852 €) en la franja baja.

**Validación de flags comerciales**: los cinco flags se han generado correctamente. Su suma (7.936) supera el total de filas (6.457) porque el grupo K se incluye simultáneamente en `perfil_rural` y `perfil_precio_sensible`, lo cual es coherente con el diseño analítico (las áreas pasivas rurales son a la vez rurales y sensibles a precio). El flag `perfil_precio_sensible` cubre el 60,8% de los códigos postales (3.923 CPs), mientras que `perfil_premium` cubre el 15,3% (989 CPs).

### Cobertura efectiva sobre la cartera de Selmark

El cruce con `silver.dim_cliente` arroja un resultado sustancialmente mejor que el estimado inicialmente en el notebook 02:

| Métrica | Valor | Porcentaje |
|---|---|---|
| Clientes totales | 3.469 | 100,0 % |
| Clientes nacionales | 1.869 | 53,9 % |
| Nacionales con MOSAIC | 1.820 | 97,4 % de los nacionales |

**Conclusión metodológica**: el 97,4% de la cartera nacional dispone de enriquecimiento MOSAIC, lo que valida la decisión de centrar el análisis de geomarketing en este segmento. Solo 49 clientes nacionales (2,6%) carecen de información MOSAIC, probablemente por tratarse de códigos postales nuevos o residuales no cubiertos por la herramienta. Este volumen es despreciable y no compromete la representatividad del análisis.

### Decisiones metodológicas

**Reducción a 12 columnas finales**. Se han descartado las 64 columnas restantes de bronze (proporciones por segmento fino) por no aportar valor analítico al alcance del TFG. Estas columnas se conservan en bronze y están disponibles para análisis avanzados en caso necesario.

**Flags comerciales con solapamiento intencional**. Los flags `perfil_rural` y `perfil_precio_sensible` se diseñan deliberadamente para incluir ambos al grupo K, ya que las áreas rurales pasivas comparten ambas características (zona rural y sensibilidad al precio). Esta redundancia analítica facilita filtrados específicos en notebooks posteriores.

**Tratamiento de los nulos en `renta_media`**. Los 626 valores nulos no se imputan en Silver para preservar la trazabilidad del dato original. La decisión sobre imputación (mediana provincial frente a exclusión) se tomará en `gold.cliente_360` según el análisis específico que requiera la variable.

### Cuestiones pendientes con el tutor

1. **Año de referencia de los datos MOSAIC**: relevante para evaluar la coherencia temporal con la ventana de análisis 2022-2025.
2. **Documentación oficial de Experian**: confirmar el acceso a la guía completa "Tipologías Mosaic V.5" para enriquecer la sección interpretativa de la memoria.

### Próximos pasos

Con la finalización de este notebook se cierra la fase Silver del proyecto. Las cinco tablas Silver disponibles (`dim_cliente`, `fact_lineas_pedido`, `ventas_minoristas`, `tiempo`, `mosaic`) constituyen la base sobre la que se construirá `gold.cliente_360`. Antes de iniciar la fase Gold, se realizará una sesión de revisión de las dudas acumuladas para el tutor, con el objetivo de confirmar las decisiones metodológicas pendientes que afectan al diseño de la tabla cliente final.

In [10]:
con.close()
print("✅ Conexión cerrada. silver.mosaic guardada en disco.")
print("\n🎉 FASE SILVER COMPLETADA — 5 de 5 tablas construidas:")
print("   ✅ silver.dim_cliente")
print("   ✅ silver.fact_lineas_pedido")
print("   ✅ silver.ventas_minoristas")
print("   ✅ silver.tiempo")
print("   ✅ silver.mosaic")

✅ Conexión cerrada. silver.mosaic guardada en disco.

🎉 FASE SILVER COMPLETADA — 5 de 5 tablas construidas:
   ✅ silver.dim_cliente
   ✅ silver.fact_lineas_pedido
   ✅ silver.ventas_minoristas
   ✅ silver.tiempo
   ✅ silver.mosaic
